In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# # STEP 1: Install dependencies
# !pip install pdfplumber transformers datasets accelerate sentencepiece

# # STEP 2: Extract text from PDF
# import pdfplumber

# pdf_path = "/content/drive/MyDrive/Dataset/train_your_bot.pdf"
# text_data = []
# with pdfplumber.open(pdf_path) as pdf:
#     for page in pdf.pages:
#         text_data.append(page.extract_text())

# full_text = "\n".join([t for t in text_data if t])

# # ===============================
# # STEP 3: Chunk the text
# # ===============================
# def chunk_text(text, chunk_size=500):
#     words = text.split()
#     for i in range(0, len(words), chunk_size):
#         yield " ".join(words[i:i+chunk_size])

# chunks = list(chunk_text(full_text, chunk_size=120))  # small chunks for LLM

# print(f"Total chunks: {len(chunks)}")
# print("Sample chunk:\n", chunks[0][:300])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 114.1 MB/s eta 0:00:00


Total chunks: 323
Sample chunk:
 Methodology: writing it up? Project Report • title page (project title; your name; client name) Abstract • an abstract, Intro • Introduction (project aims) • a literature survey Body • A description of methods used • Analysis & discussion of project outcomes • Conclusions Conclusion • Recommendation


In [ ]:
# !pip install -q transformers accelerate

# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch

# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype=torch.float16
# )

# def generate_response(prompt, max_new_tokens=150):
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         temperature=0.7,
#         do_sample=True
#     )
#     return tokenizer.decode(outputs[0], skip_special_tokens=True)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# # STEP 5: Generate dataset with periodic saving
# import json
# import os

# dataset = []
# save_path = "/content/drive/MyDrive/Dataset/dataset.jsonl"
# save_interval = 20   # <-- save after every 20 chunks (you can change this)

# # If file exists from earlier run, load it so we continue appending
# if os.path.exists(save_path):
#     with open(save_path, "r", encoding="utf-8") as f:
#         dataset = [json.loads(line) for line in f]
#     print(f"Resuming from existing dataset with {len(dataset)} entries.")

# for i, chunk in enumerate(chunks[:100]):
#     # 1) Summarization
#     summary_prompt = f"Summarize this text:\n{chunk}"
#     summary = generate_response(summary_prompt)
#     dataset.append({
#         "instruction": "Summarize the following text.",
#         "input": chunk,
#         "output": summary
#     })

#     # 2) Q&A
#     qa_prompt = f"Generate one question and answer from this text:\n{chunk}"
#     qa = generate_response(qa_prompt)
#     dataset.append({
#         "instruction": "Answer the following question based on the text.",
#         "input": chunk,
#         "output": qa
#     })

#     # 3) MCQ
#     mcq_prompt = f"Create one multiple-choice question with 3 options (A, B, C) and the correct answer from this text:\n{chunk}"
#     mcq = generate_response(mcq_prompt)
#     dataset.append({
#         "instruction": "Choose the correct option.",
#         "input": chunk,
#         "output": mcq
#     })

#     # Save at intervals
#     if (i + 1) % save_interval == 0 or (i + 1) == len(chunks[:150]):
#         with open(save_path, "w", encoding="utf-8") as f:
#             for entry in dataset:
#                 f.write(json.dumps(entry, ensure_ascii=False) + "\n")
#         print(f"✅ Saved {len(dataset)} entries at chunk {i+1}")

# print("Sample generated dataset entry:\n", dataset[0])


✅ Saved 60 entries at chunk 20
✅ Saved 120 entries at chunk 40
✅ Saved 180 entries at chunk 60
✅ Saved 240 entries at chunk 80
✅ Saved 300 entries at chunk 100
Sample generated dataset entry:
 {'instruction': 'Summarize the following text.', 'input': 'Methodology: writing it up? Project Report • title page (project title; your name; client name) Abstract • an abstract, Intro • Introduction (project aims) • a literature survey Body • A description of methods used • Analysis & discussion of project outcomes • Conclusions Conclusion • Recommendations • Lessons Learned The FINAL project report also requires a description of methods used. Project approach - Methodology …. a methodology or project approach should describe how you are going to conduct your project. This may be as simple as a series of steps Or it may be a recognised methodology in an area of Information Technology. Last week we looked at the concept of methodology in terms of your literature review and', 'output': 'Summarize 

In [ ]:
# # ===============================
# # STEP 6: Save dataset
# # ===============================
# with open("dataset.jsonl", "w", encoding="utf-8") as f:
#     for entry in dataset:
#         f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# print(" Dataset saved as dataset.jsonl")


In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 12.6 MB/s eta 0:00:00


## Loading the model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer


model_name="tiiuae/falcon-7b-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_falcon.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b-instruct:
- configuration_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.



modeling_falcon.py: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Let's also load the tokenizer below

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

In [ ]:
# from datasets import load_dataset

# jsonl_path = "/content/finetune_data.jsonl"  # <-- change path if needed
# dataset = load_dataset("json", data_files=jsonl_path, split="train")
# def preprocess(example):
#     text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"

#     toks = tokenizer(
#         text,
#         truncation=True,
#         max_length=512,
#         padding="max_length"
#     )
#     toks["labels"] = toks["input_ids"].copy()
#     return toks

# tokenized_dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

# print("Tokenized sample:\n",tokenized_dataset[0])

In [ ]:
from datasets import load_dataset

jsonl_path = "/content/drive/MyDrive/Dataset/dataset.jsonl"  # <-- path to your dataset file
dataset = load_dataset("json", data_files=jsonl_path, split="train")

def preprocess(example):
    # Build the prompt text in instruct style
    if example["input"].strip():
        # Case: when input exists
        text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        # Case: no input (direct Q&A style)
        text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

    toks = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    toks["labels"] = toks["input_ids"].copy()
    return toks

tokenized_dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

print("Tokenized sample:\n", tokenized_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenized sample:
 {'input_ids': [19468, 31498, 37, 193, 45630, 270, 907, 248, 1863, 2288, 25, 1001, 19468, 21495, 37, 193, 10394, 1303, 37, 2607, 334, 510, 42, 5010, 6150, 204, 2291, 4120, 2113, 204, 19, 14265, 4120, 38, 402, 1536, 38, 3637, 1536, 20, 20617, 204, 2291, 267, 12192, 23, 52397, 204, 2291, 17568, 204, 19, 14265, 10806, 20, 204, 2291, 241, 9231, 6414, 10451, 204, 2291, 317, 6485, 275, 4300, 1042, 204, 2291, 10348, 204, 17, 5568, 275, 1441, 8073, 204, 2291, 64611, 37362, 204, 2291, 37484, 204, 2291, 30353, 53012, 390, 29274, 2040, 1441, 1416, 614, 4654, 241, 6485, 275, 4300, 1042, 25, 5010, 3213, 204, 24, 60447, 204, 5689, 241, 18539, 379, 1441, 3213, 808, 7474, 672, 299, 362, 1089, 271, 3838, 402, 1441, 25, 735, 724, 314, 345, 2299, 345, 241, 2559, 275, 4128, 1515, 334, 724, 314, 241, 21490, 18539, 272, 267, 1766, 275, 4639, 6391, 25, 6570, 1356, 360, 2998, 388, 248, 3722, 275, 18539, 272, 3072, 275, 402, 9231, 1856, 273, 1001, 19468, 16054, 37, 193, 45630, 270, 907, 414, 

Below we will load the configuration file in order to create the LoRA model. According to QLoRA paper, it is important to consider all linear layers in the transformer block for maximum performance. Therefore we will add `dense`, `dense_h_to_4_h` and `dense_4h_to_h` layers in the target modules in addition to the mixed query key value layer.

In [ ]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 32           #64 cilo

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "query_key_value",
        "dense",
        "dense_h_to_4h",
        "dense_4h_to_h",
    ]
)

## Loading the trainer

## **trainer**

In [ ]:
from transformers import TrainingArguments

output_dir = "./results"
per_device_train_batch_size = 2  #4 cilo
gradient_accumulation_steps = 4
optim = "paged_adamw_32bit"
save_steps = 10
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 250
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    gradient_checkpointing=True,
)

Then finally pass everthing to the trainer

In [ ]:
from trl import SFTTrainer

max_seq_length = 512

trainer = SFTTrainer(
   model= model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    # dataset_text_field="text",
    # max_seq_length=max_seq_length,
    # tokenizer=tokenizer,
    args=training_arguments,
)

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


Truncating train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

We will also pre-process the model by upcasting the layer norms in float 32 for more stable training

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

## Train the model

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: raihantanvir118 (raihantanvir118-rajshahi-university-of-engineering-techn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.731100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [ ]:
output_dir = "./results"
final_path = f"{output_dir}/final_checkpoint" # This evaluates to "./results/final_checkpoint"
trainer.model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

('./results/final_checkpoint/tokenizer_config.json',
 './results/final_checkpoint/special_tokens_map.json',
 './results/final_checkpoint/chat_template.jinja',
 './results/final_checkpoint/tokenizer.json')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import upload_folder

repo_id = "tanvir211/falcon-7b-lora-finetuned"
upload_folder(
    folder_path=final_path,
    repo_id=repo_id
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 25.4kB /  261MB            

CommitInfo(commit_url='https://huggingface.co/tanvir211/falcon-7b-lora-finetuned/commit/51eefe7122b5819ff316e555569a12cb8736125e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='51eefe7122b5819ff316e555569a12cb8736125e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tanvir211/falcon-7b-lora-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='tanvir211/falcon-7b-lora-finetuned'), pr_revision=None, pr_num=None)